# Data Preparation

## Preparation Strategy

The objective of this notebook is to prepare the raw data for analysis by assessing and improving its quality.

The preparation process follows a structured workflow that focuses on:

- Duplicate records
- Missing values
- Data types
- Business consistency

Each identified issue is evaluated based on its impact on the business analysis.

Issues that affect the analytical results are cleaned or corrected, while issues that do not influence the planned analysis are documented and intentionally left unchanged.

The output of this notebook is a clean and analysis-ready dataset for the exploratory analysis and dashboard development stages.

## Load Raw Data

The raw source tables are loaded again in this notebook to ensure that the data preparation process is independent and reproducible.

No changes are made to the original files stored in the `data/raw` directory.

In [1]:
# =====================================
# Import Libraries
# =====================================

import pandas as pd
from IPython.display import display

In [2]:
# =====================================
# Load Raw Business Tables
# =====================================

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [3]:
# =====================================
# Store Raw Tables
# =====================================

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

## Duplicate Record Assessment

This section checks each raw table for fully duplicated rows.

Duplicate records may distort counts, aggregations, and business metrics. At this stage, duplicates are only identified and documented before any removal decision is made.

In [4]:
# =====================================
# Assess Fully Duplicate Rows
# =====================================

duplicate_summary = []

for table_name, df in tables.items():
    duplicate_count = df.duplicated().sum()
    #print(duplicate_count)

    duplicate_summary.append({
        "table_name": table_name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count
    })
   

duplicate_summary = pd.DataFrame(duplicate_summary)

display(duplicate_summary)

,table_name,total_rows,duplicate_rows
0,customers,99441,0
1,orders,99441,0
2,order_items,112650,0
3,payments,103886,0
4,reviews,99224,0
5,products,32951,0
6,sellers,3095,0
7,geolocation,1000163,261831
8,category_translation,71,0


### Finding

No fully duplicated rows were found in the core business tables used for the analysis.

The `geolocation` table contains a large number of duplicated rows. Since this table is not used in the current analytical workflow, no cleaning action is applied at this stage.

## Missing Value Assessment

This section evaluates missing values across the dataset.

Each missing value is assessed based on its business meaning and potential impact on the planned analysis.

Cleaning decisions are made only when the missing values affect analytical results.

### Orders Table

The Orders table contains several missing values related to the order lifecycle timestamps.

Each missing value is evaluated to determine whether it represents a valid business scenario or a data quality issue.

In [5]:
# =====================================
# Inspect Missing Values in Orders
# =====================================

orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [6]:
# =====================================
# Orders with Missing Approval Date
# =====================================

missing_approval = orders[orders["order_approved_at"].isnull()]

missing_approval["order_status"].value_counts()

# missing_approval["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [7]:
# =====================================
# Orders with Missing Carrier Date
# =====================================

missing_carrier = orders[orders["order_delivered_carrier_date"].isnull()]

missing_carrier["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [8]:
# =====================================
# Orders with Missing Customer Delivery Date
# =====================================

missing_customer_delivery = orders[orders["order_delivered_customer_date"].isnull()]

missing_customer_delivery["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [9]:
# =====================================
# Delivered Orders with Missing Dates
# =====================================

delivered_orders = orders[orders["order_status"] == "delivered"]

delivered_missing_dates = delivered_orders[
    delivered_orders[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].isnull().any(axis=1)
]

delivered_missing_dates

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


#### Findings and Decision

- Most missing order timestamps are consistent with the related order status.
- A small number of delivered orders contain missing timestamps and are treated as data-quality issues.
- These values will not be imputed. The affected records will only be excluded from analyses that require the missing date.

### Products Table

The Products table contains missing values in product description and dimension-related attributes.

Each missing value is assessed to determine whether it affects the planned business analysis.

In [10]:
# =====================================
# Inspect Missing Values in Products
# =====================================

products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [11]:
# =====================================
# Products with Missing Category
# =====================================

missing_product_category = products[products["product_category_name"].isnull()]

missing_product_category[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isnull().sum()

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

In [12]:
# =====================================
# Products with Missing Dimensions
# =====================================

missing_dimensions = products[products["product_weight_g"].isnull()]

missing_dimensions

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Findings and Decision

The same 610 products are missing category and product metadata.

Two products also contain missing weight and dimension values.

These values will not be imputed because they cannot be inferred reliably. The affected records will only be excluded from analyses that require the missing attributes.

### Reviews Table

The Reviews table contains missing values in customer review comments.

Each missing value is assessed to determine whether it affects the planned business analysis.

In [13]:
# =====================================
# Inspect Missing Values in Reviews
# =====================================

reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [14]:
# =====================================
# Reviews Without Title and Comment
# =====================================

empty_reviews = reviews[
    reviews["review_comment_title"].isnull() &
    reviews["review_comment_message"].isnull()
]

len(empty_reviews)

56518

#### Findings and Decision

A large number of reviews contain missing titles and/or comments.

The analysis shows that many customers submitted only a review score without providing textual feedback. Therefore, these missing values are considered expected user behaviour rather than data-quality issues.

The missing values will remain unchanged and will only be excluded from analyses that require textual review content.

## Missing Value Summary

### Summary

- Missing order timestamps were assessed based on the order lifecycle.
- Missing product metadata was identified as incomplete product information.
- Missing review titles and comments were determined to be expected user behaviour.

### Overall Decision

No missing values were imputed during data preparation.

Records containing missing values will remain unchanged and will only be excluded from analyses that require the affected attributes.

## Data Type Validation

This section validates whether each column has the appropriate data type for analysis.

Columns with incorrect data types are converted to their appropriate formats before further analysis.

In [15]:
# =====================================
# Check Date Columns
# =====================================

orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

### Findings and Decision

All order date columns are currently stored as the `object` data type.

Since these columns represent timestamps, they will be converted to the `datetime` data type to support time-based analysis.

In [16]:
# =====================================
# Convert Date Columns to Datetime
# =====================================

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)

orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

In [17]:
# =====================================
# Validate Date Columns
# =====================================

orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### Findings and Decision

The order date columns were stored as `object` and were successfully converted to `datetime64[ns]`.

## Business Consistency Validation


This section validates whether the prepared data follows the expected business rules required for the planned business analysis.

Only business rules that directly affect the project KPIs are evaluated.

### Rule 1: Approval After Purchase

**Business Rule**

An order must be approved on or after the purchase timestamp.

**Validation**

This rule checks whether any orders were approved before they were purchased.

In [18]:
# ===================================
# Approval After Purchase
# ===================================

invalid_approval = orders[
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
]

len(invalid_approval)

0

### Findings
No records were found where an order was approved before it was purchased.

The purchase and approval timestamps follow the expected business sequence.

### Rule 2: Delivery After Purchase

**Business Rule**

A customer cannot receive an order before placing it.

**Validation**

This rule checks whether any customer delivery timestamps occur before the purchase timestamp.

In [19]:
# =====================================
# Delivery After Purchase
# =====================================

invalid_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

len(invalid_delivery)

0

#### Findings

No records were found where an order was delivered before it was purchased.

The purchase and delivery timestamps follow the expected business sequence.

## Business Consistency Summary

### Summary

The evaluated business rules are consistent with the expected order lifecycle.

No business rule violations were identified that could affect the planned business analysis.

# Data Preparation Summary

## Summary

The data preparation process included duplicate assessment, missing value assessment, data type validation, and business consistency validation.

No duplicate records requiring removal were identified. Missing values were assessed based on business context, and only the required data type conversions were applied. Business consistency checks confirmed that the prepared data follows the expected order lifecycle.

The dataset is now ready for exploratory data analysis.